# Airbnb neighbourhood geometry (GeoJSON) Bronze ingestion (F07.2-T02 / T03)

Ingests the Toronto Inside Airbnb `neighbourhoods.geojson` into its own
Bronze data product (`airbnb_neighbourhood_geometry_bronze`) - a
SEPARATE asset from the neighbourhoods CSV product; the two
representations are never combined. Grain is ONE ROW PER FEATURE: the
observed property keys (`neighbourhood`, `neighbourhood_group`) as raw
strings plus `geometry_json` carrying each feature's FULL geometry
object as a compact JSON string - spatial information is never
flattened away.

The GeoJSON structure is validated before parsing (FeatureCollection,
canonical property/geometry fingerprint against the registered
schema_version), and the feature count is recorded on the manifest
row's `loaded_row_count` and asserted by the raw==bronze reconciliation
(F07.2-T03). Features missing a name or geometry are RETAINED with
`_quality_status` / `_quality_reason` - never dropped.

The complete ingestion flow lives ONCE in `src/airbnb/bronze_flow.py`;
this notebook supplies only the geometry dataset spec, the run widgets
and the result display. City, snapshot date and source family are
derived from the landed manifest record itself - never hard-coded.

In [0]:
# Run-mode controls (F02.2 / F05.3 conventions, applied to the
# neighbourhood geometry dataset in F07.2-T02/T03).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved into the dedicated trippilot_test catalog through
# src.common.environment. "live" resolves the development tables and is
# used only after the test-mode evidence gate has passed.
#
# force_reload is the explicit reprocess authorization: it permits an
# exact rerun of an already-ingested file and the controlled replacement
# of a corrected source file (a different source_file_id).
#
# source_file_id pins the EXACT landed manifest record this run reads:
# seven files share the family/entity/version identity
# insideairbnb/toronto/2026-09-16 (detailed listings/calendar/reviews
# csv.gz + summary listings/reviews csv + neighbourhoods csv/geojson),
# so resolving by identity alone is ambiguous - the explicit id is
# required.
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"])
dbutils.widgets.text("force_reload", "false")
dbutils.widgets.text(
    "source_file_id",
    "7b7f77a66b5bb1d109a635159166b321603991486bf75729569fde8d0e167f73",
)

RUN_MODE = dbutils.widgets.get("run_mode")
FORCE_RELOAD = dbutils.widgets.get("force_reload")
SOURCE_FILE_ID = dbutils.widgets.get("source_file_id")

print(f"Run mode: {RUN_MODE} | force_reload: {FORCE_RELOAD}")
print(f"source_file_id: {SOURCE_FILE_ID}")

In [0]:
# Locate the repository root, then run the shared flow (F07.2-T02).
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the TripPilot repository root (src/ and config/)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.airbnb.bronze_flow import BronzeDatasetSpec, run_bronze_ingestion

# Dataset identity for the neighbourhood geometry path (F07.1
# evidence): the shared flow derives city/snapshot/source from the
# landed manifest record and parses one row per GeoJSON feature.
GEOMETRY_SPEC = BronzeDatasetSpec(
    dataset="neighbourhood_geometry",
    schema_label="neighbourhood-geojson",
    contract_path="contracts/airbnb/bronze_neighbourhood_geometry.yml",
    rules_key="bronze_neighbourhood_geometry",
    bronze_key="neighbourhood_geometry_bronze",
    default_source_file_id="7b7f77a66b5bb1d109a635159166b321603991486bf75729569fde8d0e167f73",
)

summary = run_bronze_ingestion(
    GEOMETRY_SPEC,
    spark,
    source_file_id=SOURCE_FILE_ID,
    run_mode=RUN_MODE,
    force_reload=FORCE_RELOAD,
)

In [0]:
# Result display: F07.2 acceptance evidence (raw sample, retained
# invalid rows, per-identity verification, per-slice summary).
display(summary["raw_df"].limit(5))

if summary["invalid_rows"]:
    from pyspark.sql import functions as F

    display(
        summary["bronze_df"].filter(F.col("_quality_status") == "invalid").limit(20)
    )

display(summary["verification_df"])
display(summary["snapshot_summary"])